In [4]:
import sys
from collections import Counter
from pathlib import Path

import spacy
import unicodedataplus as ud

In [3]:
def scriptScore(token, targetScript: str = 'ARABIC'):
    """
    Returns a score from 0.0 to 1.0 representing the proportion
    of characters in 'token' belonging to 'targetScript'.
    """
    if not token:
        return 0.0

    matchCount = 0
    # Normalize target script to uppercase for comparison
    targetScript = targetScript.upper()

    ArabicVowels = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652"
    for ch in token:
        script = ud.script(ch).upper() if ch not in ArabicVowels else 'ARABIC'
        if script in (targetScript, "COMMON"):
            matchCount += 1
        else:
            print(ch, script)

    return matchCount / len(token)

In [4]:
ArabicVowels = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652"

for ch in ArabicVowels:
    print(f"'{ch}': {scriptScore(ch)} - {ud.script(ch).upper()}")

'ً': 1.0 - INHERITED
'ٌ': 1.0 - INHERITED
'ٍ': 1.0 - INHERITED
'َ': 1.0 - INHERITED
'ُ': 1.0 - INHERITED
'ِ': 1.0 - INHERITED
'ّ': 1.0 - INHERITED
'ْ': 1.0 - INHERITED


In [5]:
ud.script(" ")

'Common'

In [6]:
def tokenize(text: str):
    nlp = spacy.blank('ur')

    # Spacy based tokenization
    doc = nlp(text)
    return  doc

In [7]:
def tokens2Vocab(doc, isAlpha: bool = True, scoreThreshold: float = 0.9) -> tuple[Counter, int]:

    words = [token.text for token in doc if (isAlpha and token.is_alpha) and scriptScore(token.text) >= scoreThreshold]
    wordCount = len(words)

    vocab = Counter(words)

    return vocab, wordCount

In [8]:
urduText = """
فرض کرو ہم تارے ہوتے
ايک دوجے کو دور دور سے ديکھ ديکھ کر جلتے بجتے
اور پھر ايک دن
شاخِ فلک سے گرتے اور تاريک خلاؤں ميں کھو جاتے
دريا کے دو دھارے ہوتے،
اپنى اپنى موج ميں بہتے
اور سمندر تک اس اَندھى، وحشى اور منہ زور مسافت
کے جادو ميں تنہا رہتے
فرض کررو ہم بھور سمے کے پنچھى ہوتے،
اُڑتے اُڑتے ايک دوجے کو چھوتے اور پھر
کھلے گگن کى گہرى اور بے صرفہ آنکھوں ميں کھو جاتے
اور بہار کے جھونکے ہوتے،
موسم کے اک بےنقشہ خواب ميں ملتے
ملتے اور جدا ہو جاتے
خشک زمينوں کے ہاتھوں پر سبز لکيريں کندہ کرتے
اور ان ديکھے سپنے بوتے
اپنے اپنے رو کر چين سے سو جاتے
فرض کرو ہم جو کچھ اب ہيں وہ ناں ہوتے۔۔۔
"""

In [8]:
txt = Path(r"training_Urdu_PK.txt").read_text(encoding='utf8')
if len(txt) > 1_000_000:
    txt = txt[:1_000_000]
    print("Trimming input text size to 1,000,000")
else:
    print(f"Input text size: {len(txt):,}")

Input text size: 119,238


In [9]:
doc = tokenize(txt)
tokenCount = len(doc)
vocabulary, wordCount = tokens2Vocab(doc)
vocabCount = len(vocabulary)
print(f"Totals:: {tokenCount=:,} {wordCount=:,} {vocabCount=:,}")


Totals:: tokenCount=28,639 wordCount=25,295 vocabCount=4,041


In [10]:
print(f"{'Word':<15} | Frequency | Score")
print("-" * 25)
for w, f in vocabulary.most_common():
    s = scriptScore(w)
    if s < 1.0 :
        print(f"{w:15} | {f:9} | {s}")

print(f"\n{'Word':<15} | Frequency n(%) | Score")
print("-" * 25)
for w, f in vocabulary.most_common(n=100):
    s = scriptScore(w)
    print(f"{w:15} | {f:9}({(f/wordCount)*100:0.2f}%) | {s}")

Word            | Frequency | Score
-------------------------

Word            | Frequency n(%) | Score
-------------------------
کے              |       989(3.91%) | 1.0
کی              |       841(3.32%) | 1.0
ہے              |       829(3.28%) | 1.0
میں             |       800(3.16%) | 1.0
اور             |       728(2.88%) | 1.0
کا              |       548(2.17%) | 1.0
سے              |       547(2.16%) | 1.0
ہیں             |       384(1.52%) | 1.0
اس              |       363(1.44%) | 1.0
کو              |       285(1.13%) | 1.0
نے              |       280(1.11%) | 1.0
ان              |       262(1.04%) | 1.0
بھی             |       228(0.90%) | 1.0
کہ              |       197(0.78%) | 1.0
ایک             |       187(0.74%) | 1.0
کر              |       169(0.67%) | 1.0
پر              |       169(0.67%) | 1.0
و               |       167(0.66%) | 1.0
یہ              |       165(0.65%) | 1.0
نہیں            |       165(0.65%) | 1.0
کیا             |       163(0.64%) | 1.0
وہ       

In [26]:
Words5kTxt = Path(r"Urdu5k.txt").read_text(encoding='utf8')
lines = [line.strip() for line in Words5kTxt.splitlines()]
len(lines)

10000

In [24]:
vocab = []
i = 0
while i < len(lines):
    l1 = scriptScore(lines[i]) == 1.0
    l2 = scriptScore(lines[i+1]) == 1.0
    if l1 and l2:
        w = lines[i]
        f = lines[i+1]
        assert f.isdigit(), f"{i+1} line is not freq"
        vocab.append((w, f))
    else:
        print(lines[i], lines[i+1])
    i += 2

In [25]:
len(vocab)

5000

In [41]:
for i, l in enumerate(lines, start=1):
    if i % 2 == 0:
        # print(i, l)
        assert l.strip().isdigit(), f"{i} line is not numeric"
    else:
        assert scriptScore(l.strip()) == 1.0

In [31]:
lines[:]

['ﮐﮯ',
 '743949',
 'ﻣﻴﮟ',
 '582882',
 'ﮐﯽ',
 '575545',
 'ﮨﮯ',
 '466908',
 'اور',
 '413788']

In [80]:
CORRECT_URDU_CHARACTERS: dict = {'آ': ['ﺁ', 'ﺂ'],
                                 'أ': ['ﺃ'],
                                 'ا': ['ﺍ', 'ﺎ', ],
                                 'ب': ['ﺏ', 'ﺐ', 'ﺑ', 'ﺒ'],
                                 'پ': ['ﭖ', 'ﭗ', 'ﭘ', 'ﭙ'],
                                 'ت': ['ﺕ', 'ﺖ', 'ﺗ', 'ﺘ'],
                                 'ٹ': ['ﭦ', 'ﭧ', 'ﭨ', 'ﭩ'],
                                 'ث': ['ﺛ', 'ﺜ', 'ﺚ'],
                                 'ج': ['ﺝ', 'ﺞ', 'ﺟ', 'ﺠ'],
                                 'ح': ['ﺡ', 'ﺣ', 'ﺤ', 'ﺢ'],
                                 'خ': ['ﺧ', 'ﺨ', 'ﺦ'],
                                 'د': ['ﺩ', 'ﺪ'],
                                 'ذ': ['ﺬ', 'ﺫ'],
                                 'ر': ['ﺭ', 'ﺮ'],
                                 'ز': ['ﺯ', 'ﺰ', ],
                                 'س': ['ﺱ', 'ﺲ', 'ﺳ', 'ﺴ', ],
                                 'ش': ['ﺵ', 'ﺶ', 'ﺷ', 'ﺸ'],
                                 'ص': ['ﺹ', 'ﺺ', 'ﺻ', 'ﺼ', ],
                                 'ض': ['ﺽ', 'ﺾ', 'ﺿ', 'ﻀ'],
                                 'ط': ['ﻃ', 'ﻄ', 'ﻂ'],
                                 'ظ': ['ﻅ', 'ﻇ', 'ﻆ', 'ﻈ'],
                                 'ع': ['ﻉ', 'ﻊ', 'ﻋ', 'ﻌ', ],
                                 'غ': ['ﻍ', 'ﻏ', 'ﻐ', 'ﻎ'],
                                 'ف': ['ﻑ', 'ﻒ', 'ﻓ', 'ﻔ', ],
                                 'ق': ['ﻕ', 'ﻖ', 'ﻗ', 'ﻘ', ],
                                 'ل': ['ﻝ', 'ﻞ', 'ﻟ', 'ﻠ', ],
                                 'م': ['ﻡ', 'ﻢ', 'ﻣ', 'ﻤ', ],
                                 'ن': ['ﻥ', 'ﻦ', 'ﻧ', 'ﻨ', ],
                                 'چ': ['ﭺ', 'ﭻ', 'ﭼ', 'ﭽ'],
                                 'ڈ': ['ﮈ', 'ﮉ'],
                                 'ڑ': ['ﮍ', 'ﮌ'],
                                 'ژ': ['ﮋ', ],
                                 'ک': ['ﮎ', 'ﮏ', 'ﮐ', 'ﮑ', 'ﻛ', 'ك'],
                                 'گ': ['ﮒ', 'ﮓ', 'ﮔ', 'ﮕ'],
                                 'ں': ['ﮞ', 'ﮟ'],
                                 'و': ['ﻮ', 'ﻭ', 'ﻮ', ],
                                 'ؤ': ['ﺅ'],
                                 'ھ': ['ﮪ', 'ﮬ', 'ﮭ', 'ﻬ', 'ﻫ', 'ﮫ'],
                                 'ہ': ['ﻩ', 'ﮦ', 'ﻪ', 'ﮧ', 'ﮩ', 'ﮨ', 'ه', ],
                                 'ۂ': [],
                                 'ۃ': ['ة'],
                                 'ء': ['ﺀ'],
                                 'ی': ['ﯼ', 'ى', 'ﯽ', 'ﻰ', 'ﻱ', 'ﻲ', 'ﯾ', 'ﯿ', 'ي', 'ﻳ', 'ﻴ'],
                                 'ئ': ['ﺋ', 'ﺌ', ],
                                 'ے': ['ﮮ', 'ﮯ' ],
                                 'ۓ': [],
                                 '۰': ['٠'],
                                 '۱': ['١'],
                                 '۲': ['٢'],
                                 '۳': ['٣'],
                                 '۴': ['٤'],
                                 '۵': ['٥'],
                                 '۶': ['٦'],
                                 '۷': ['٧'],
                                 '۸': ['٨'],
                                 '۹': ['٩'],
                                 '۔': [],
                                 '؟': [],
                                 '٫': [],
                                 '،': [],
                                 'لا': ['ﻻ', 'ﻼ', 'ﻵ'],
                                 '': ['ـ']

                                 }

_TRANSLATOR = {}
for key, value in CORRECT_URDU_CHARACTERS.items():
    _TRANSLATOR.update(dict.fromkeys(map(ord, value), key))

In [3]:
def getUniRep(txt):
    assert txt is not None
    assert type(txt) == str

    uc = [f"U+{ord(c):04X}" for c in txt]
    return " ".join(uc[::-1])

In [10]:
def isArabicBlock(text):
    """Checks if all characters are within the main Arabic Unicode block."""
    return all('\u0600' <= char <= '\u06FF' for char in text if not char.isspace())

In [82]:
with open(r"Urdu5k.sym", "w", encoding="utf-8") as dic:
    for i, (w, f) in enumerate(vocab):
        ww = w.translate(_TRANSLATOR)
        if not isArabicBlock(ww):
            print(i, getUniRep(w), w)
            ww = w.translate(_TRANSLATOR)
            print(i, getUniRep(ww), ww, '\n')
        dic.write(f"{ww}${f}\n")

In [68]:
w, f = vocab[1427]
print(getUniRep(w), w)
ww = w.translate(_TRANSLATOR)
print(getUniRep(ww), ww)

U+06BA U+FEEE U+FEE4 U+FEF4 U+FEC8 U+FEE8 U+FE97 ﺗﻨﻈﻴﻤﻮں
U+06BA U+0648 U+0645 U+06CC U+0638 U+0646 U+062A تنظیموں


In [1]:
def normalize2Urdu(token: str) -> str:
    """
    Standardizes Urdu tokens by:
    1. Mapping positional variants (Initial/Medial/Final forms) to base characters.
    2. Converting Arabic/Persian range characters to Urdu standard block.
    3. Removing diacritics and non-spacing marks.
    """
    if not token:
        return token

    # 1. Compatibility Decomposition (NFKC)
    # This automatically converts most positional variants (e.g., ﻒ, ﻘ, ﻂ)
    # from the Presentation Forms blocks to their standard Arabic script bases.
    token = ud.normalize('NFKC', token)

    # 2. Urdu-Specific Base Character Mapping
    # After NFKC, some chars might be in the 'Arabic' block (0643).
    # We must force them into the 'Urdu' preferred block.
    urduBaseMapping = {
        # Kaf variants
        '\u0643': '\u06a9', # Arabic Kaf -> Urdu Kaf
        '\u06a8': '\u06a9', # Swash Kaf -> Urdu Kaf

        # Yeh variants
        '\u064a': '\u06cc', # Arabic Yeh -> Urdu Chooti Yeh
        '\u0649': '\u06cc', # Alif Maqsura -> Urdu Chooti Yeh
        '\u06d2': '\u06d2', # Preserve Bari Yeh

        # Heh variants
        '\u0647': '\u06c1', # Arabic Heh -> Urdu Gol Heh
        '\u0629': '\u06c1', # Ta Marbuta -> Urdu Gol Heh

        # Zero-Width non-joiners (often used in positional variants)
        '\u200c': '',
    }

    for target, replacement in urduBaseMapping.items():
        token = token.replace(target, replacement)

    # 3. Strip Diacritics (Zabar, Zer, Pesh, etc.)
    # We use NFD to isolate marks, then filter them out.
    nfd_form = ud.normalize('NFD', token)
    token = "".join([c for c in nfd_form if not ud.combining(c)])

    return ud.normalize('NFC', token)

In [11]:
# Example: Testing with a 'Medial' form variant
# Character 'ﻒ' (Final Fe) becomes 'ف'
testToken = "\ufeef\u064e" # Final Fe + Zabar
print(f"Original: {testToken} [{getUniRep(testToken)}]")
tt = normalize2Urdu(testToken)
print(f"Normalized: {tt} [{getUniRep(tt)}]")

Original: ﻯَ [U+064E U+FEEF]
Normalized: ی [U+06CC]


In [14]:
URDU_VARIANT_MAP = {
    '\u0627': ['\u0627', '\uFE8D', '\uFE8E'],  # Alif
    '\u0628': ['\u0628', '\uFE8F', '\uFE90', '\uFE91', '\uFE92'],  # Be
    '\u067E': ['\u067E', '\uFB56', '\uFB57', '\uFB58', '\uFB59'],  # Pe
    '\u062A': ['\u062A', '\uFE95', '\uFE96', '\uFE97', '\uFE98'],  # Te
    '\u0679': ['\u0679', '\uFB66', '\uFB67', '\uFB68', '\uFB69'],  # Tte
    '\u062B': ['\u062B', '\uFE99', '\uFE9A', '\uFE9B', '\uFE9C'],  # Se
    '\u062C': ['\u062C', '\uFE9D', '\uFE9E', '\uFE9F', '\uFEA0'],  # Jeem
    '\u0686': ['\u0686', '\uFB7A', '\uFB7B', '\uFB7C', '\uFB7D'],  # Che
    '\u062D': ['\u062D', '\uFEA1', '\uFEA2', '\uFEA3', '\uFEA4'],  # Bari He
    '\u062E': ['\u062E', '\uFEA5', '\uFEA6', '\uFEA7', '\uFEA8'],  # Khe
    '\u062F': ['\u062F', '\uFEA9', '\uFEAA'],  # Dal
    '\u0688': ['\u0688', '\uFB88', '\uFB89'],  # Ddal
    '\u0630': ['\u0630', '\uFEAB', '\uFEAC'],  # Zal
    '\u0631': ['\u0631', '\uFEAD', '\uFEAE'],  # Re
    '\u0691': ['\u0691', '\uFB8C', '\uFB8D'],  # Rre
    '\u0632': ['\u0632', '\uFEAF', '\uFEB0'],  # Ze
    '\u0698': ['\u0698', '\uFB8A', '\uFB8B'],  # Zhe
    '\u0633': ['\u0633', '\uFEB1', '\uFEB2', '\uFEB3', '\uFEB4'],  # Seen
    '\u0634': ['\u0634', '\uFEB5', '\uFEB6', '\uFEB7', '\uFEB8'],  # Sheen
    '\u0635': ['\u0635', '\uFEB9', '\uFEBA', '\uFEBB', '\uFEBC'],  # Suad
    '\u0636': ['\u0636', '\uFEBD', '\uFEBE', '\uFEBF', '\uFEC0'],  # Zuad
    '\u0637': ['\u0637', '\uFEC1', '\uFEC2', '\uFEC3', '\uFEC4'],  # To'e
    '\u0638': ['\u0638', '\uFEC5', '\uFEC6', '\uFEC7', '\uFEC8'],  # Zo'e
    '\u0639': ['\u0639', '\uFEC9', '\uFECA', '\uFECB', '\uFECC'],  # Ain
    '\u063A': ['\u063A', '\uFECD', '\uFECE', '\uFECF', '\uFED0'],  # Ghain
    '\u0641': ['\u0641', '\uFED1', '\uFED2', '\uFED3', '\uFED4'],  # Fe
    '\u0642': ['\u0642', '\uFED5', '\uFED6', '\uFED7', '\uFED8'],  # Qaf
    '\u06A9': ['\u06A9', '\uFB8E', '\uFB8F', '\uFB90', '\uFB91',   # Urdu Kaf (Urdu/Persian)
               '\u0643', '\uFED9', '\uFEDA', '\uFEDB', '\uFEDC',   # Urdu Kaf (Arabic)
               '\u06A8', '\uFB96', '\uFB97', '\uFB98', '\uFB99'],  # Urdu Kaf (Old Persian)
    '\u06AF': ['\u06AF', '\uFB92', '\uFB93', '\uFB94', '\uFB95'],  # Gaf
    '\u0644': ['\u0644', '\uFEDD', '\uFEDE', '\uFEDF', '\uFEE0'],  # Lam
    '\u0645': ['\u0645', '\uFEE1', '\uFEE2', '\uFEE3', '\uFEE4'],  # Meem
    '\u0646': ['\u0646', '\uFEE5', '\uFEE6', '\uFEE7', '\uFEE8'],  # Noon
    '\u06BA': ['\u06BA', '\uFB9E', '\uFB9F'],  # Noon Ghunna
    '\u0648': ['\u0648', '\uFEED', '\uFEEE',   # Wao
               '\u0624', '\uFE85', '\uFE86'],  # Wao with Hamza
    '\u06C1': [
        '\u06C1', '\uFBA6', '\uFBA7', '\uFBA8', '\uFBA9',  # Urdu Heh Goal & Positional
        '\u0647', '\uFEE9', '\uFEEA', '\uFEEB', '\uFEEC',  # Arabic Ha & Positional
        '\u06C2', '\u06C0',                                # Heh with Hamza variants
        '\u0629', '\uFE93', '\uFE94'                       # Te Marbuta variants
    ],
    '\u06BE': ['\u06BE', '\uFBAC', '\uFBAD', '\uFBAE', '\uFBAF'],  # Do Chashmi He
    '\u06CC': ['\u06CC', '\uFBFB', '\uFBFC', '\uFBFD', '\uFBFE',   # Choti Ye (Farsi/Urdu)
               '\u064A', '\uFEF1', '\uFEF2', '\uFEF3', '\uFEF4',   # Choti Ye (Arabic/Sindhi)
               '\u0649', '\uEEF1', '\uEEF2'],                      # Choti Ye (Arabic Alef Maksura)
    '\u06D2': ['\u06D2', '\uFBAE', '\uFBAF'],  # Bari Ye
}

URDU_REVERSAL_LOOKUP = {v: base for base, variants in URDU_VARIANT_MAP.items() for v in variants}

def normalizeUrduChars(text):
    """
    Replaces presentation/positional forms with standard Urdu tokens
    using the built-in map function for performance.
    """
    if not text:
        return ""

    # map() applies the lambda to every character in the string.
    # .get(c, c) ensures we keep characters not in our dictionary (like spaces/punctuation).
    return "".join(map(lambda c: URDU_REVERSAL_LOOKUP.get(c, c), text))

In [13]:
testToken = 'ﺗﻨﻈﻴﻤﻮں'
print(f"Original: {testToken} [{getUniRep(testToken)}]")
tt = normalizeUrduChars(testToken)
print(f"Normalized: {tt} [{getUniRep(tt)}]")

Original: ﺗﻨﻈﻴﻤﻮں [U+06BA U+FEEE U+FEE4 U+FEF4 U+FEC8 U+FEE8 U+FE97]
Normalized: تنظیموں [U+06BA U+0648 U+0645 U+06CC U+0638 U+0646 U+062A]


In [24]:
def removeMarks(text):
    # Selective Diacritic Removal (NFD)
    # NFD is used to isolate marks, then filter them out.
    # Marks to preserve:
    # U+0653 (Madda - for آ)
    # U+0654 (Hamza Above - for ئ),
    # other base characters are filtered by character normalization)
    preservedMarks = {'\u0653', '\u0654'}
    nfdForm = ud.normalize('NFD', text)
    token = "".join([c for c in nfdForm if c in preservedMarks or not ud.combining(c)])
    return ud.normalize('NFC', token)

def removePunctuation(text):
    return "".join([c for c in text if not ud.category(c).startswith('P')])

def removeDigits(text):
    return "".join([c for c in text if not ud.category(c) == 'Nd'])

def normalizeNonChars(text):
    return removePunctuation(removeDigits(removeMarks(text)))

def normalizeWhiteSpace(text):
    return  " ".join(text.split())

In [28]:
testToken = 'ـ'
print(f"Original: {testToken} [{getUniRep(testToken)}]")
tt = normalizeWhiteSpace(normalizeNonChars(testToken))
print(f"Normalized: {tt} [{getUniRep(tt)}]")

Original: ـ [U+0640]
Normalized: ـ [U+0640]
